## illegal fishing police patrol analysis

What we plan on accomplishing is the following:
1. [Load dataset onto Notebook instance from S3](#Step-1:-Loading-the-data-from-Amazon-S3)
2. [Cleaning, transforming, and preparing the data](#Step-2:-Cleaning,-transforming,-and-preparing-the-data)
3. [Create and train our model](#Step-3:-Create-and-train-our-model)
4. [Viewing the results](#Step-4:-Viewing-the-results)
5. Visualize using QuickSight

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

import boto3
from sagemaker import get_execution_role
import sagemaker.amazon.common as smac

## Step 1: Loading the data from Amazon S3

In [2]:
role = get_execution_role()
bucket = 'alexbc.datalake'
prefix = 'illegal-activity'
data_key = 'fishing_dataset.csv'
data_location = 's3://{}/{}/{}'.format(bucket, prefix, data_key)

df = pd.read_csv(data_location, low_memory=False)

In [3]:
df.head()

,reportedTimestamp,eventDate,eventTime,shape,duration,witnesses,weather,latitude,longitude,naturalDamage,legalEvidence,caseSituation,crimeOutcome
0,1977-04-04T04:02:23.340Z,1977-03-31,23:46,cargo,4,1,rain,47.329444,-122.578889,Y,N,N,illegal
1,1982-11-22T02:06:32.019Z,1982-11-15,22:04,boat,4,1,partly cloudy,52.664913,-1.034894,Y,Y,N,illegal
2,1992-12-07T19:06:52.482Z,1992-12-07,19:01,cargo,49,1,clear,38.951667,-92.333889,Y,N,N,illegal
3,2011-02-24T21:06:34.898Z,2011-02-21,20:56,boat,13,1,partly cloudy,41.496944,-71.367778,Y,N,N,illegal
4,1991-03-09T16:18:45.501Z,1991-03-09,11:42,cargo,17,1,mostly cloudy,47.606389,-122.330833,Y,N,N,illegal


In [4]:
df.shape

(18000, 13)

## Step 2: Cleaning, transforming, and preparing the data
Create another DataFrame with just the latitude and longitude attributes

In [5]:
df_geo = df[['latitude', 'longitude']]

In [6]:
df_geo.head()

,latitude,longitude
0,47.329444,-122.578889
1,52.664913,-1.034894
2,38.951667,-92.333889
3,41.496944,-71.367778
4,47.606389,-122.330833


In [7]:
df_geo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18000 entries, 0 to 17999
Data columns (total 2 columns):
latitude     18000 non-null float64
longitude    18000 non-null float64
dtypes: float64(2)
memory usage: 281.3 KB


In [8]:
missing_values = df_geo.isnull().values.any()
print('Are there any missing values? {}'.format(missing_values))
if(missing_values):
    df_geo[df_geo.isnull().any(axis=1)]

Are there any missing values? False


Transform the pandas DataFrame (our dataset) into a numpy.ndarray. When we do this each row is converted to a Record object. According to the documentation, this is what the K-Means algorithm expects as training data. This is what we will use as training data for our model.

[See the documentation for input training](https://sagemaker.readthedocs.io/en/stable/kmeans.html)

In [9]:
data_train = df_geo.values.astype('float32')
data_train

array([[  47.329445, -122.57889 ],
       [  52.664913,   -1.034894],
       [  38.951668,  -92.333885],
       ...,
       [  36.86639 ,  -83.888885],
       [  35.385834,  -94.39833 ],
       [  29.883055,  -97.94111 ]], dtype=float32)

## Step 3: Create and train our model
In this step we will import and use the built-in SageMaker K-Means algorithm. We will set the number of cluster to 10 (for our 10 sensors), specify the instance type we want to train on, and the location of where we want our model artifact to live. 

[See the documentation of hyperparameters here](https://docs.aws.amazon.com/sagemaker/latest/dg/k-means-api-config.html)

In [10]:
from sagemaker import KMeans

num_clusters = 20
output_location = 's3://' + bucket + '/model-artifacts'

kmeans = KMeans(role=role,
               train_instance_count=1,
               train_instance_type='ml.c4.xlarge',
               output_path=output_location,
               k=num_clusters)

In [11]:
job_name = 'kmeans-geo-job-{}'.format(datetime.now().strftime("%Y%m%d%H%M%S"))
print('Here is the job name {}'.format(job_name))

Here is the job name kmeans-geo-job-20191012163435


In [12]:
%%time
kmeans.fit(kmeans.record_set(data_train), job_name=job_name)

2019-10-12 16:34:37 Starting - Starting the training job...
2019-10-12 16:34:39 Starting - Launching requested ML instances......
2019-10-12 16:36:07 Starting - Preparing the instances for training.........
2019-10-12 16:37:11 Downloading - Downloading input data...
2019-10-12 16:38:08 Training - Training image download completed. Training in progress.
2019-10-12 16:38:08 Uploading - Uploading generated training model
2019-10-12 16:38:08 Completed - Training job completed
Docker entrypoint called with argument(s): train
[10/12/2019 16:37:58 INFO 139769815770944] Reading default configuration from /opt/amazon/lib/python2.7/site-packages/algorithm/resources/default-input.json: {u'_enable_profiler': u'false', u'_tuning_objective_metric': u'', u'_num_gpus': u'auto', u'local_lloyd_num_trials': u'auto', u'_log_level': u'info', u'_kvstore': u'auto', u'local_lloyd_init_method': u'kmeans++', u'force_dense': u'true', u'epochs': u'1', u'init_method': u'random', u'local_lloyd_tol': u'0.0001', u'lo

Training seconds: 57
Billable seconds: 57
CPU times: user 697 ms, sys: 26.8 ms, total: 724 ms
Wall time: 3min 42s


## Step 4: Viewing the results
In this step we will take a look at the model artifact SageMaker created for us and stored onto S3. We have to do a few special things to see the latitude and longitude for our 10 clusters (and the center points of those clusters)

[See the documentation of deserilization here](https://docs.aws.amazon.com/sagemaker/latest/dg/cdf-training.html#td-deserialization)

At this point we need to "deserilize" the model artifact. Here we are going to open and review them in our notebook instance. We can unzip the model artifact which will contain model_algo-1. This is just a serialized Apache MXNet object. From here we can load that serialized object into a numpy.ndarray and then extract the clustered centroids from the numpy.ndarray.

After we extract the results into a DataFrame of latitudes and longitudes, we can create a CSV with that data, load it onto S3 and then visualize it with QuickSight.

In [13]:
import os
model_key = 'model-artifacts/' + job_name + '/output/model.tar.gz'

boto3.resource('s3').Bucket(bucket).download_file(model_key, 'model.tar.gz')
os.system('tar -zxvf model.tar.gz')
os.system('unzip model_algo-1')

2304

In [14]:
!pip install mxnet

You are using pip version 10.0.1, however version 19.2.3 is available.
You should consider upgrading via the 'pip install --upgrade pip' command.


In [15]:
import mxnet as mx
Kmeans_model_params = mx.ndarray.load('model_algo-1')

In [16]:
cluster_centroids_kmeans = pd.DataFrame(Kmeans_model_params[0].asnumpy())
cluster_centroids_kmeans.columns=df_geo.columns
cluster_centroids_kmeans

,latitude,longitude
0,31.722809,-97.019073
1,15.902567,115.685516
2,47.013489,-120.765518
3,52.142994,0.686745
4,33.734238,-87.372032
5,18.763674,77.562866
6,-32.109303,146.831314
7,36.926403,-109.080872
8,42.956234,-71.679268
9,41.255070,-93.628632


Let's go ahead and upload this dataset onto S3 and view within QuickSight

In [17]:
from io import StringIO

results_location = '{}/{}'.format(prefix, "locations_kmeans.csv")

csv_buffer = StringIO()
cluster_centroids_kmeans.to_csv(csv_buffer, index=False)

s3_resource = boto3.resource('s3')
s3_resource.Object(bucket, results_location).put(Body=csv_buffer.getvalue())

{'ResponseMetadata': {'RequestId': 'A67DEDEAEF2CE487',
  'HostId': 'KI+AIUiqVOKKStvhQHugmAlKC8djkvuJhdDrYXowp4pq1AzAaSfNsdJnxL8i1L+KgnonkBggRuo=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'KI+AIUiqVOKKStvhQHugmAlKC8djkvuJhdDrYXowp4pq1AzAaSfNsdJnxL8i1L+KgnonkBggRuo=',
   'x-amz-request-id': 'A67DEDEAEF2CE487',
   'date': 'Sat, 12 Oct 2019 16:38:40 GMT',
   'etag': '"69fb762284ed81270c753b558a9f0951"',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"69fb762284ed81270c753b558a9f0951"'}